## ChatMessageHistory的使用

内存存储：ChatMessageHistory 默认是在内存中运行的，重启程序数据就会丢失。如果需要永久存储，请使用 SQLChatMessageHistory 或 RedisChatMessageHistory 等持久化实现。

#### 场景1:记忆存储

In [36]:
from langchain_classic.chains.conversation.base import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_classic.chains.summarize.refine_prompts import prompt_template
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory
# ChatMessageHistory的实例化
from langchain_community.chat_message_histories import ChatMessageHistory, RedisChatMessageHistory
from openai import conversations

# 初始化一个内存记录器
history = ChatMessageHistory()

# 添加消息
history.add_user_message('你好，我叫曹雪芹')
history.add_user_message('你好，郭德纲有什么可以帮你的')

# 获取记录
print(history.messages)


[HumanMessage(content='你好，我叫曹雪芹', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，郭德纲有什么可以帮你的', additional_kwargs={}, response_metadata={})]


#### 场景2:对接LLM

In [38]:
# 方式一：基础用法
from langchain_openai import ChatOpenAI
import os
import dotenv
from langchain_community.chat_message_histories import ChatMessageHistory

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

history = ChatMessageHistory()

# history.add_user_message('你好，我叫曹雪芹')
# history.add_user_message('你好，郭德纲有什么可以帮你的')
history.add_user_message('我叫什么名字')

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)

res = model.invoke(history.messages)
print(res.content)


我无法查看或获取您的个人信息，包括您的名字。为了保护您的隐私和安全，我不会询问或尝试获取您的个人身份信息。如果您愿意，您可以直接告诉我您的名字，我会记住并使用它来称呼您。


## 自动注入上下文RunnableWithMessageHistory

In [20]:
# 方式二:自动注入上下文
'''
MessagesPlaceholder 是构建“多轮对话模板”的核心，它让你在模板中预留一个插入历史对话的位置。
RunnableWithMessageHistory 负责管理历史记录的存取、注入和更新，让你无需手动拼接历史。
这种模式非常适合构建聊天机器人、客服系统等需要上下文记忆的应用。

在开发阶段，ChatMessageHistory 非常方便。但上线前，请务必替换为 RedisChatMessageHistory 等持久化方案，防止服务重启导致数据丢失。
'''
from langchain_openai import ChatOpenAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv

# RedisChatMessageHistory()

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)
# 1. 构建一个带历史占位符的提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system","你是一位友好的助手"),
    MessagesPlaceholder(variable_name="history"),  # 历史消息占位符
    ("human", "{input}")  # 当前用户输入
])

# 2. 构建链：prompt → model
chain = prompt | model

# 用于存储不同 session_id 的历史记录 (实际中可以用 Redis/Postgres 替代)
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 包装你的模型
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 调用
config = {"configurable": {"session_id": "session1"}}
response = chain_with_history.invoke({"input": "你好，我叫小明"}, config=config)
print('AI回复：',response.content)
response1=chain_with_history.invoke({"input": "毕业于哈佛学院"}, config=config)
print('AI回复：',response1.content)
# 下次调用时，模型会自动记住“我叫小明”
response2 = chain_with_history.invoke({"input": "小明毕业于什么学校？"}, config=config)
print('AI回复：',response2.content)

print('-----')
print(store["session1"])
# 输出: 你叫小明。

AI回复： 你好，小明！很高兴认识你。有什么我可以帮助你的吗？
AI回复： 哇，那可真是个了不起的成就！哈佛学院的教育质量非常高，你在那里的学习经历一定非常丰富。你在哈佛主修什么专业呢？有没有什么特别难忘的经历或收获？
AI回复： 小明，你刚刚提到自己毕业于哈佛学院，对吗？那是一个非常著名的学府。你在哈佛的学习和生活一定有很多精彩的故事吧？
-----
Human: 你好，我叫小明
AI: 你好，小明！很高兴认识你。有什么我可以帮助你的吗？
Human: 毕业于哈佛学院
AI: 哇，那可真是个了不起的成就！哈佛学院的教育质量非常高，你在那里的学习经历一定非常丰富。你在哈佛主修什么专业呢？有没有什么特别难忘的经历或收获？
Human: 小明毕业于什么学校？
AI: 小明，你刚刚提到自己毕业于哈佛学院，对吗？那是一个非常著名的学府。你在哈佛的学习和生活一定有很多精彩的故事吧？


## ConversationBufferMemory的使用（已经过期，不在维护）

In [9]:
# 举例一
from langchain_classic.memory.buffer import ConversationBufferMemory

memory = ConversationBufferMemory()
# inputs
memory.save_context(inputs={'human': '你好，我是Tony老师'}, outputs={'ai': '不高兴认识你'})
memory.save_context(inputs={'input': '帮我回答一下1+2*3=？'}, outputs={'output': 'so easy等于7'})

print(memory.load_memory_variables({}))

print('\n')
print('------')
print(memory.chat_memory.messages)


{'history': 'Human: 你好，我是Tony老师\nAI: 不高兴认识你\nHuman: 帮我回答一下1+2*3=？\nAI: so easy等于7'}


------
[HumanMessage(content='你好，我是Tony老师', additional_kwargs={}, response_metadata={}), AIMessage(content='不高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我回答一下1+2*3=？', additional_kwargs={}, response_metadata={}), AIMessage(content='so easy等于7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


## 使用LangGraph

In [6]:
# 举例二：推荐使用LangGraph（也可以使用上面的自动注入）
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langgraph.graph.message import MessagesState  # 正确的导入
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
import os
import dotenv

# 1. 定义节点：调用 LLM 的函数
dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)


def call_model(state: MessagesState):
    # state["messages"] 是历史消息列表
    response = model.invoke(state["messages"])
    return {"messages": [response]}  # 必须返回字典，包含新增的消息


# 2. 构建图
# InMemorySaver 会将历史消息自动保存在内存中，通过 thread_id 区分不同会话。
memory = InMemorySaver()
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node("chatbot", call_model)

# 添加边：入口点 -> chatbot -> 结束
builder.set_entry_point("chatbot")
builder.set_finish_point("chatbot")

# 编译图，传入 checkpointer
graph = builder.compile(checkpointer=memory)

# 3. 调用
config = {"configurable": {"thread_id": "1"}}
# 注意：消息必须是 BaseMessage 对象列表
input_message = HumanMessage(content="你好,我叫迪迦,我喜欢吃棒棒糖")
response = graph.invoke({"messages": [input_message]}, config=config)

# 输出 AI 回复
print(response["messages"][-1].content)

# 第二次调用（自动记住历史）
input_message2 = HumanMessage(content="我叫什么名字？回答名字就行")
response2 = graph.invoke({"messages": [input_message2]}, config=config)
# 输出 AI 回复
print(response2["messages"][-1].content)
# 第三次调用（自动记住历史）
input_message3 = HumanMessage(content='我喜欢吃什么？')
response3 = graph.invoke({"messages": [input_message3]}, config=config)

print(response3["messages"][-1].content)

你好，迪迦！很高兴认识你。棒棒糖确实很美味，甜甜的，五颜六色的，让人看了心情都会变好。你最喜欢什么口味的棒棒糖呢？
迪迦
棒棒糖


## ConversationChain(方法已经过期了，不在维护)依然使用上面的自动注入

## ConversationBufferWindowMemory(方法已经过期了，不在维护)依然使用上面的自动注入

In [27]:
# 举例：结合llm、chain的使用
from langchain_classic.memory.buffer_window import ConversationBufferWindowMemory
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
# 自定义提示词模版
prompt_template=PromptTemplate.from_template(
    "以下是人类与AI之间的对话，AI表现的很健谈，并提供了大量来自上下文的具体细节,如果AI不知道问题的答案，它会表示不知道，当前对话{history}Human:{question}AI:''"
)
# 创建大模型
llm=ChatOpenAI(
     model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)
# 示例好，设置窗口阀值
memory=ConversationBufferWindowMemory(k=1)

# 定义LLMChain
conversation_with_summary=LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    # 日志记录
    # verbose=True
)
# 执行链
res1=conversation_with_summary.invoke({"question":"你好，我的名字是孙悟空"})

res2=conversation_with_summary.invoke({"question":"我还有三个徒弟"})

res3=conversation_with_summary.invoke({"question":"我今年100岁了"})

res4=conversation_with_summary.invoke({"question":"我叫什么名字?"})

print(res4)



{'question': '我叫什么名字?', 'history': 'Human: 我今年100岁了\nAI: 哇，100岁了！这真是一个了不起的年龄，充满了智慧和经验。在这一百年里，你一定见证了许多历史性的时刻，经历了无数的变迁。能否分享一下你这一生中印象最深刻的一段经历？或者，是什么让你保持了如此长久的活力和好奇心呢？', 'text': '我目前无法确定您的名字，因为对话中没有提供相关信息。如果您愿意，可以告诉我您的名字，或者分享一些关于您自己的信息，我会尽力记住并称呼您。'}


In [4]:
# 现在的替代方案：trim_messages

'''
你可以根据 Token 数量 而不是仅仅根据 消息条数 来进行修剪，且支持保留最后几条消息的同时保留系统提示词
优点：
    1、更智能的修剪：旧版的 BufferWindowMemory 只能数消息条数，完全不顾及每条消息的长度（可能 1 条消息就占了 10k Token）。trim_messages 可以精确控制 Token 总量，防止超长上下文带来的额外成本。
    2、保留 System Prompt：在旧版中，如果你不小心，WindowMemory 可能会把你的系统提示词（Persona）给删掉。现代的 trim_messages 允许你设置 include_system=True，确保核心人设永远不会丢失。
    3、调试性：这是 LCEL 的一部分，你可以随时在日志里看到修剪前后的消息列表，不再是藏在 Memory 对象内部的黑盒。
'''
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages.utils import count_tokens_approximately
import os
import dotenv


dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)

# 1. 定义修剪器 (取代 WindowMemory)
# 这会保持最近的 10 条消息，如果超过 Token 限制则会自动切除
trimmer = trim_messages(
    max_tokens=20000,
    strategy="last",
    # token_counter=model，即让模型自身去计算。
    # 有些模型LangChain 还没为这个模型适配 Token 计数功能。
    token_counter=count_tokens_approximately,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

# 2. 定义 Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个非常友好的助手。"),
    MessagesPlaceholder(variable_name="history"),  # 这里会动态填入历史
    ("human", "{input}"),
])

# 3. 组合链 (将修剪逻辑作为链的一环)
chain = (
        RunnablePassthrough.assign(
            history=lambda x: trimmer.invoke(x["history"])
        )
        | prompt
        | model
)
# 配合 RunnableWithMessageHistory 使用即可
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
# 调用
config = {"configurable": {"session_id": "session1"}}
response = chain_with_history.invoke({"input": "你好，我叫小明"}, config=config)
response1 = chain_with_history.invoke({"input": "我叫什么名字？"}, config=config)
print(response1.content)

你叫小明。
